# 🌸 Ejercicio: Clasificación de Iris con MLflow

## 🎯 Objetivos del Ejercicio

En este ejercicio práctico aprenderás a:

1. **Trabajar con un problema de clasificación** multiclase
2. **Aplicar MLflow** a un modelo de árbol de decisión
3. **Registrar experimentos** de forma profesional
4. **Evaluar y comparar** diferentes configuraciones
5. **Crear visualizaciones** para clasificación

---

## 🌺 El Dataset Iris

El **Iris Dataset** es uno de los datasets más famosos en Machine Learning, introducido por Ronald Fisher en 1936.

### 📊 Características del Dataset

| Aspecto | Descripción |
|---------|-------------|
| **Muestras** | 150 flores (50 por clase) |
| **Características** | 4 medidas físicas (cm) |
| **Clases** | 3 especies de iris |
| **Tipo** | Clasificación multiclase |
| **Dificultad** | ⭐⭐ (Principiante) |

### 🌸 Las 3 Especies de Iris

1. **Iris Setosa** (Clase 0)
2. **Iris Versicolor** (Clase 1)
3. **Iris Virginica** (Clase 2)

### 📏 Las 4 Características

1. **Sepal Length** (Longitud del sépalo)
2. **Sepal Width** (Ancho del sépalo)
3. **Petal Length** (Longitud del pétalo)
4. **Petal Width** (Ancho del pétalo)

---

## 🎯 Tu Misión

Construir un **Árbol de Decisión** que pueda clasificar correctamente las especies de iris basándose en sus medidas físicas, y documentar todo el proceso con MLflow.

---

## 🚀 ¡Empecemos!

In [0]:
import mlflow
import warnings
warnings.filterwarnings('ignore')


mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

# ⚠️ IMPORTANTE: Cambia esto por tu email de Databricks
email = '202616729@alu.comillas.edu'  # Ejemplo: 'nombre.apellido@ejemplo.com'

# Validar que el email no esté vacío
if not email:
    print("⚠️  ADVERTENCIA: Debes configurar tu email antes de continuar")
    print("   Cambia la variable 'email' por tu email de Databricks")
else:
    # Configurar el experimento
    experiment_name = f"/Users/{email}/4-ejercicio-the-irish"
    mlflow.set_experiment(experiment_name)
    
    print("=" * 60)
    print("✅ MLflow configurado correctamente")
    print("=" * 60)
    print(f"📊 Experimento: {experiment_name}")
    print(f"🌸 Dataset: Iris (Clasificación)")
    print(f"🤖 Modelo: Decision Tree Classifier")
    print("=" * 60)

## ⚙️ Paso 1: Configuración de MLflow

Configuramos el experimento donde se registrarán todas las ejecuciones.

**🔴 IMPORTANTE**: Cambia el email vacío por tu email de Databricks.

## 📚 Paso 2: Importar Librerías

Importamos todas las herramientas necesarias para clasificación.

In [0]:
%pip install seaborn

In [0]:
%restart_python

In [0]:
# Librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# MLflow para tracking
import mlflow
import mlflow.sklearn

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report,
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Todas las librerías importadas correctamente")
print("   - MLflow: Listo para tracking")
print("   - Scikit-learn: Modelos y métricas")
print("   - Matplotlib & Seaborn: Visualizaciones")

## 🌺 Paso 3: Cargar y Explorar el Dataset Iris

Cargaremos el famoso dataset de flores Iris y exploraremos sus características.

In [0]:
# TODO: Carga el dataset Iris y explora su estructura.
# Pistas:
# - Usa datasets.load_iris()
# - Crea un DataFrame con dataset.data y dataset.feature_names
# - Añade la columna target/especie
# - Muestra shape, clases, primeras filas y estadísticas descriptivas

dataset = datasets.load_iris()
df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
df['species'] = dataset.target
df.head()


In [0]:
df.describe()

### 📊 Visualización Exploratoria del Dataset

Visualicemos las relaciones entre las características para entender mejor los datos.

## Relación entre longitud/ancho del pétalo por especie

In [0]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df, 
    x='petal length (cm)', 
    y='petal width (cm)', 
    hue='species',     # Color por especia
    palette='Set2',   
    s=70               # Tamaño de los puntos
)

plt.title('Relación entre Longitud y Ancho del Pétalo por Especie', fontsize=14, pad=15)
plt.xlabel('Longitud del Pétalo (cm)', fontsize=12)
plt.ylabel('Ancho del Pétalo (cm)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6) # Malla de fondo

plt.show()

Se observa que en especies como `setosa` y `versicolor` hay una relación casi 1:1 entre la longitud del ancho y el pétalo, mientras que en `virginica` hay más dispersión. Aun así, podríamos proponer un modelo de regresión lineal para las tres.

## Distribución de características por especie

In [0]:
species_names = {0: 'setosa', 1: 'versicolor', 2: 'virginica'}
df['species_name'] = df['species'].map(species_names)

# Creamos un lienzo de 2x2 para las 4 características independientes
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Boxplot para Sepal Length
sns.boxplot(data=df, x='species_name', y='sepal length (cm)', hue='species_name', palette='Set2', ax=axes[0, 0], legend=False)
axes[0, 0].set_title('Distribución de Sepal Length')

# 2. Boxplot para Sepal Width
sns.boxplot(data=df, x='species_name', y='sepal width (cm)', hue='species_name', palette='Set2', ax=axes[0, 1], legend=False)
axes[0, 1].set_title('Distribución de Sepal Width')

# 3. Boxplot para Petal Length
sns.boxplot(data=df, x='species_name', y='petal length (cm)', hue='species_name', palette='Set2', ax=axes[1, 0], legend=False)
axes[1, 0].set_title('Distribución de Petal Length')

# 4. Boxplot para Petal Width
sns.boxplot(data=df, x='species_name', y='petal width (cm)', hue='species_name', palette='Set2', ax=axes[1, 1], legend=False)
axes[1, 1].set_title('Distribución de Petal Width')

plt.tight_layout()
plt.show()



## Matriz de correlación

In [0]:
# 1. Seleccionar solo las columnas numéricas (características del dataset)
columnas_numericas = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
matriz_corr = df[columnas_numericas].corr()

# 2. Configurar el lienzo de la gráfica
plt.figure(figsize=(8, 6))

# 3. Crear el Heatmap
sns.heatmap(
    matriz_corr, 
    annot=True,           # Muestra el número de correlación en cada celda
    cmap='coolwarm',      # Escala de color: azul (negativo), blanco (neutro), rojo (positivo)
    fmt=".2f",            # Redondear a 2 decimales
    linewidths=0.5,       # Espaciado fino entre celdas
    vmin=-1, vmax=1       # Forzar que la escala vaya de -1 a 1
)

# Añadir título
plt.title('Matriz de Correlación de las Características de Iris', fontsize=14, pad=15)
plt.show()


Según Petal length y petal width, se obserba separabilidad, al contrario que en las otras dos variables.
En cuanto a la correlación, se observa una fuerte correlación de petal width y petal length, lo que indica que estos pétalos podrían ser cuadrados.

## 🔀 Paso 4: Preparar los Datos

Dividimos el dataset en conjuntos de entrenamiento y prueba.

In [0]:
df.head()

In [0]:
# TODO: Separa características (X) y etiquetas (y).
variables_predictoras = [
    'sepal length (cm)',
    'sepal width (cm)',
    'petal length (cm)',
    'petal width (cm)'
]
X = df[variables_predictoras]
y = df['species']

RANDOM_STATE = 42

# TODO: Divide el dataset en entrenamiento y prueba.
# Pistas:
# - Usa train_test_split
# - Reserva una parte para test
# - Usa random_state para reproducibilidad
# - Considera stratify=y para mantener la proporción de clases

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y  # estratificamos la variable objetivo
)

# TODO: Comprueba tamaños y distribución de clases en cada conjunto.

X_train.shape, X_test.shape, y_train.shape, y_test.shape


## 🌳 Paso 5: Entrenar Árbol de Decisión con MLflow

Ahora entrenaremos un modelo de **Decision Tree** y registraremos todo con MLflow.

### 🔑 Hiperparámetros del Árbol de Decisión

- **max_depth**: Profundidad máxima del árbol (evita overfitting)
- **max_features**: Número máximo de características a considerar por split
- Valores más altos = modelo más complejo = mayor riesgo de overfitting

In [0]:
# ========================================
# TODO: ENTRENAMIENTO CON MLFLOW
# ========================================

# La IA me ha añadido esto para quitar warnings de MLflow
import logging
logging.getLogger('mlflow.tracking.context.registry').setLevel(logging.ERROR)

# TODO: Activa autologging de MLflow.
mlflow.sklearn.autolog()

# TODO: Inicia un run con un nombre descriptivo.
with mlflow.start_run(run_name="Decision Tree - Iris Classifier") as run:
#     TODO: Define hiperparámetros del árbol.
    max_depth = 5
    max_features = 4
    random_state = RANDOM_STATE
#
#     TODO: Crea y entrena DecisionTreeClassifier.
    dt = DecisionTreeClassifier(max_depth=max_depth, max_features=max_features, random_state=random_state)
    dt.fit(X_train, y_train)
#
#     TODO: Genera predicciones para train y test.
    y_pred_train = dt.predict(X_train)
    y_pred_test = dt.predict(X_test)
#
#     TODO: Calcula métricas de clasificación.
    y_pred_test = dt.predict(X_test)
    accuracy_test = accuracy_score(y_test, y_pred_test)

    precision_test = precision_score(y_test, y_pred_test, average='weighted')
    recall_test = recall_score(y_test, y_pred_test, average='weighted')
    f1_test = f1_score(y_test, y_pred_test, average='weighted')
#
#     TODO: Evalúa con cross-validation.
    cv_scores = cross_val_score(dt, X_train, y_train, cv=5)
#
#     TODO: Registra métricas/parámetros relevantes en MLflow si no quedan registrados automáticamente.
    mlflow.log_metric("accuracy_test", accuracy_test)
    mlflow.log_metric("precision_test", precision_test)
    mlflow.log_metric("recall_test", recall_test)
    mlflow.log_metric("f1_test", f1_test)
    mlflow.log_metric("cv_scores_mean", cv_scores.mean())
#
#     TODO: Visualiza matriz de confusión y, opcionalmente, el árbol.
    confusion_matrix(y_test, dt.predict(X_test))
#


In [0]:
plot_tree(dt);

## 🎯 Reflexión del Ejercicio

Completa esta sección cuando termines el notebook.

### ✅ Comprueba que has trabajado

1. [ ] Exploración del dataset Iris
2. [ ] Separación train/test
3. [ ] Entrenamiento de un Árbol de Decisión
4. [ ] Evaluación con métricas de clasificación
5. [ ] Registro del experimento en MLflow
6. [ ] Interpretación de resultados

---

## 📚 Conceptos Clave - Clasificación

### 🎯 Métricas de Clasificación

| Métrica | Descripción | Cuándo Usarla |
|---------|-------------|---------------|
| **Accuracy** | % de predicciones correctas | Clases balanceadas |
| **Precision** | % de positivos correctos | Importante evitar falsos positivos |
| **Recall** | % de positivos encontrados | Importante encontrar todos los positivos |
| **F1-Score** | Media armónica de P y R | Balance entre precisión y recall |

### 🌳 Árbol de Decisión

**TODO: Explica con tus palabras:**
- ¿Qué ventajas tiene un árbol de decisión? EL modelo tiene mucha explicabilidad.
- ¿Qué riesgos tiene respecto a overfitting? Si ponemos un límite muy alto de profundidad, puede tener overfitting, aproximando cada dato a una hojaa, prácticamente.
- ¿Cómo interpretarías una matriz de confusión multiclase?

---

## 🚀 Desafíos Adicionales

### 🎯 Desafío 1: Optimiza los Hiperparámetros

Prueba diferentes configuraciones y encuentra la mejor:

```python
# Experimenta con:
max_depth = [3, 5, 10, None]
max_features = [2, 3, 4, 'sqrt']
min_samples_split = [2, 5, 10]
```

**TODO:** ¿Qué combinación da el mejor balance entre accuracy y overfitting?

### 🎯 Desafío 2: Implementa Grid Search

Automatiza la búsqueda del mejor modelo con `GridSearchCV`.

### 🎯 Desafío 3: Compara con Otros Modelos

Entrena y compara Random Forest, SVM, KNN o Logistic Regression.

### 🎯 Desafío 4: Análisis de Errores

**TODO:** ¿Qué flores se confunden más y por qué?

### 🎯 Desafío 5: Reducción de Dimensionalidad

Usa PCA para reducir a 2 dimensiones y visualizar límites de decisión.


In [0]:
# 🎨 DESAFÍOS OPCIONALES
# Usa este espacio para completar los desafíos del ejercicio.

# ========================================
# TODO: DESAFÍO 1 - Optimización manual
# ========================================
# configuraciones = [...]
# for config in configuraciones:
#     # Entrena, evalúa y registra cada configuración en MLflow
#     pass

max_depth_values = [3, 5, 10, None]
max_features_values = [2, 3, 4, 'sqrt']
min_samples_split_values = [2, 5, 10]

accuracys = []

for i in range(len(max_depth_values)):
    for j in range(len(max_features_values)):
        for k in range(len(min_samples_split_values)):
            mlflow.sklearn.autolog()

            with mlflow.start_run(run_name="Decision Tree - Iris Classifier") as run:

                max_depth = max_depth_values[i]
                max_features = max_features_values[j]
                min_samples_split = min_samples_split_values[k]
                random_state = RANDOM_STATE

                dt = DecisionTreeClassifier(max_depth=max_depth, max_features=max_features, random_state=random_state, min_samples_split=min_samples_split)
                dt.fit(X_train, y_train)

                y_pred_train = dt.predict(X_train)
                y_pred_test = dt.predict(X_test)

                y_pred_test = dt.predict(X_test)
                accuracy_test = accuracy_score(y_test, y_pred_test)

                accuracys.append([accuracy_test, max_depth, max_features, random_state])

print(accuracys)


In [0]:
# ========================================
# TODO: DESAFÍO 2 - Grid Search
# ========================================
# from sklearn.model_selection import GridSearchCV
# param_grid = {...}
# grid_search = GridSearchCV(...)
# grid_search.fit(...)

logging.getLogger('mlflow.tracking.context.registry').setLevel(logging.ERROR)

from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [3, 5, 10, None],
    'max_features': [2, 3, 4, 'sqrt'],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE), param_grid, cv=5)
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)
print(grid_search.best_estimator_)

In [0]:
# ========================================
# TODO: DESAFÍO 3 - Comparación de modelos
# ========================================
# modelos = {...}
# resultados = []
# for nombre, modelo in modelos.items():
#     # Entrena y compara métricas
#     pass

# Compara Random Forest, SVM, KNN o Logistic Regression

logging.getLogger('mlflow.tracking.context.registry').setLevel(logging.ERROR)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

modelos = {
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'SVM': SVC(),
    'KNN': KNeighborsClassifier(),
    'Logistic Regression': LogisticRegression()
}
resultados = []
for nombre, modelo in modelos.items():
    mlflow.sklearn.autolog()

    with mlflow.start_run(run_name=f"{nombre} - Iris Classifier") as run:
        modelo.fit(X_train, y_train)
        y_pred_test = modelo.predict(X_test)
        accuracy_test = accuracy_score(y_test, y_pred_test)
        # multiclase: hay que indicar average='weighted' (igual que en el Paso 5)
        precision_test = precision_score(y_test, y_pred_test, average='weighted')
        recall_test = recall_score(y_test, y_pred_test, average='weighted')
        f1_test = f1_score(y_test, y_pred_test, average='weighted')

        resultados.append([accuracy_test, precision_test, recall_test, f1_test])
        

df = pd.DataFrame(resultados, columns=['Accuracy', 'Precision', 'Recall', 'F1'], index=modelos.keys())
display(df)

In [0]:
# ========================================
# TODO: DESAFÍO 5 - PCA y visualización 2D
# ========================================
# from sklearn.decomposition import PCA
# pca = PCA(n_components=2)
# X_pca = ...

# Desactivar autolog de MLflow para PCA (no es un modelo predictivo)
mlflow.sklearn.autolog(disable=True)

from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
X_pca_train = pca.fit_transform(X_train)
X_pca_test = pca.transform(X_test)
